# FTS-Diffusion Rollout Debug Notebook

This notebook audits the suspected generation collapse without retraining. It loads the stored reference SISC artifacts and PGM/PEM checkpoints, then compares:

- `fixed_blocks_argmax`: author-style independent 252-day blocks from the same initial state.
- `continuous_argmax`: one continuous deterministic PEM argmax chain.
- `continuous_sampled`: a debug variant that samples the PEM pattern/length distributions instead of taking `argmax`.
- `continuous_empirical`: a sanity baseline that samples empirical next states from the train-set motif chain.

The key test is whether motif collapse is caused mainly by deterministic PEM rollout/reset logic. If so, fixed blocks should replay the same pattern-length path, and sampled/empirical transitions should restore motif diversity.

In [1]:
from __future__ import annotations

import json
import math
import os
import random
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F

ROOT = Path(globals().get("NOTEBOOK_PATH", Path.cwd())).resolve()
if ROOT.suffix == ".ipynb":
    ROOT = ROOT.parents[1]
elif not (ROOT / "fts-diffusion-ref").exists() and (ROOT.parent / "fts-diffusion-ref").exists():
    ROOT = ROOT.parent

REF = ROOT / "fts-diffusion-ref"
OUT = ROOT / "reports" / "generated_outputs" / "12_rollout_debug_notebook"
OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl")
os.chdir(REF)
sys.path.insert(0, str(REF))

from models.load_models import build_pem, build_pgm
from models.model_params import pem_params, pgm_params, prm_params
from models.sampling import segment_generation_ftsdiffusion, state_evolution_ftsdiffusion
from utils.load_data import load_segments

torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
print(f"repo root: {ROOT}")
print(f"output dir: {OUT.relative_to(ROOT)}")
print(f"torch: {torch.__version__}; device used here: cpu")

repo root: /Users/tomhaidinger/Documents/Gitlab_Projects/fts-diffusion
output dir: reports/generated_outputs/12_rollout_debug_notebook
torch: 1.13.1; device used here: cpu


/Users/tomhaidinger/miniforge3/envs/fts-diffusion/lib/python3.10/site-packages/tslearn/bases/bases.py:15: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


[cell 1 completed in 1.13s]


## Loader and Rollout Helpers

The reference loader misses the S&P 500 checkpoint filenames in this repository (`.pth.pth`), so this notebook loads state dictionaries directly while keeping the original model classes and sampling code.

In [2]:
ASSET_CONFIG = {
    "goog": {"k": 11, "label": "GOOG"},
    "sp500": {"k": 14, "label": "S&P 500"},
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def configure_asset(asset: str) -> None:
    k = ASSET_CONFIG[asset]["k"]
    prm_params.update(
        {
            "dataname": asset,
            "k": k,
            "l_min": 10,
            "l_max": 21,
            "barycenter": "dba",
            "init_strategy": "kmeans++",
        }
    )
    pgm_params.update({"n_patterns": k, "sae_custom_pad_length": 21, "pcdm_series_length": 21})
    pem_params.update({"n_patterns": k})


def checkpoint_path(kind: str, asset: str) -> Path:
    k = ASSET_CONFIG[asset]["k"]
    if kind == "pgm":
        base = f"pgm-2_c48-80_{asset}_k{k}_n30_lr4e-04_dw0.01_pw1_sw0.01"
    elif kind == "pem":
        base = f"pem_{asset}_k{k}_e196_h32_lr4e-04_pw0.05_lw0.01_mw0.94"
    else:
        raise ValueError(kind)

    model_dir = Path("trained_models")
    candidates = [
        model_dir / f"{base}.pth",
        model_dir / f"{base}.pth.pth",
        model_dir / f"{base}.pt",
        model_dir / f"{base}.pth.pt",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError((kind, asset, [str(p) for p in candidates]))


def load_reference_model(asset: str) -> dict:
    configure_asset(asset)
    device = torch.device("cpu")
    pgm = build_pgm(device)
    pem = build_pem(device)
    pgm.load_state_dict(torch.load(checkpoint_path("pgm", asset), map_location=device))
    pem.load_state_dict(torch.load(checkpoint_path("pem", asset), map_location=device))
    pgm.eval()
    pem.eval()
    return {"generation": pgm, "evolution": pem}


def load_asset_artifacts(asset: str) -> dict:
    configure_asset(asset)
    patterns, segments, labels, lengths = load_segments(asset, prm_params)
    split = int(len(segments) * 0.8)
    train_segments, test_segments = segments[:split], segments[split:]
    train_labels, test_labels = labels[:split], labels[split:]
    train_lengths, test_lengths = lengths[:split], lengths[split:]

    init_segment = test_segments[0]
    init_state = torch.tensor(
        np.stack((test_labels[0], test_lengths[0], max(init_segment) - min(init_segment)), axis=0)
    ).view(1, -1).float()
    train_states = np.stack(
        (
            train_labels.astype(int),
            train_lengths.astype(int),
            np.array([max(seg) - min(seg) for seg in train_segments], dtype=float),
        ),
        axis=1,
    )
    return {
        "patterns": patterns,
        "init_state": init_state,
        "init_segment": init_segment,
        "train_states": train_states,
        "n_segments": len(segments),
        "n_train_segments": len(train_segments),
        "n_test_segments": len(test_segments),
    }


def evolve_sampled(pem, state: torch.Tensor, l_min: int, temperature: float = 1.0) -> torch.Tensor:
    with torch.no_grad():
        pred = pem(state)
        n_patterns = pem.n_patterns
        range_length = pem.range_length
        pattern = torch.distributions.Categorical(logits=pred[:, :n_patterns] / temperature).sample().view(1, 1)
        length = (
            torch.distributions.Categorical(logits=pred[:, n_patterns : n_patterns + range_length] / temperature)
            .sample()
            .view(1, 1)
            + l_min
        )
        magnitude = pred[:, n_patterns + range_length :].float()
        return torch.cat((pattern, length, magnitude), dim=1)


def evolve_empirical(train_states: np.ndarray, state: torch.Tensor) -> torch.Tensor:
    current_pattern = int(state[0, 0].item())
    candidates = np.where(train_states[:-1, 0].astype(int) == current_pattern)[0]
    if len(candidates) == 0:
        candidates = np.arange(len(train_states) - 1)
    idx = np.random.choice(candidates)
    return torch.tensor(train_states[idx + 1]).view(1, -1).float()


def generate_one_chain(asset: str, model: dict, artifacts: dict, mode: str, days: int, seed: int, block: int = 0):
    set_seed(seed)
    configure_asset(asset)
    l_min = prm_params["l_min"]
    state = artifacts["init_state"].clone().float()
    prices = list(artifacts["init_segment"])
    rows = [
        {
            "asset": asset,
            "block": block,
            "segment_idx": 0,
            "pattern": int(state[0, 0].item()),
            "length": int(state[0, 1].item()),
            "magnitude": float(state[0, 2].item()),
        }
    ]
    while len(prices) < days:
        if mode == "argmax":
            state = state_evolution_ftsdiffusion(model["evolution"], state, l_min)
        elif mode == "sampled":
            state = evolve_sampled(model["evolution"], state, l_min)
        elif mode == "empirical":
            state = evolve_empirical(artifacts["train_states"], state)
        else:
            raise ValueError(mode)

        segment = segment_generation_ftsdiffusion(model["generation"], state, artifacts["patterns"])
        segment = segment - segment[0] + prices[-1]
        prices.extend(segment)
        rows.append(
            {
                "asset": asset,
                "block": block,
                "segment_idx": len(rows),
                "pattern": int(state[0, 0].item()),
                "length": int(state[0, 1].item()),
                "magnitude": float(state[0, 2].item()),
            }
        )
    return np.asarray(prices[:days], dtype=float), pd.DataFrame(rows)


def run_protocol(asset: str, protocol: str, days: int = 252 * 5, blocks: int = 5, seed: int = 42):
    model = load_reference_model(asset)
    artifacts = load_asset_artifacts(asset)
    if protocol == "fixed_blocks_argmax":
        price_parts = []
        state_parts = []
        for block in range(blocks):
            prices, states = generate_one_chain(asset, model, artifacts, "argmax", 252, seed + block, block=block)
            price_parts.append(prices)
            state_parts.append(states)
        return np.concatenate(price_parts), pd.concat(state_parts, ignore_index=True), artifacts

    mode = {
        "continuous_argmax": "argmax",
        "continuous_sampled": "sampled",
        "continuous_empirical": "empirical",
    }[protocol]
    prices, states = generate_one_chain(asset, model, artifacts, mode, days, seed, block=0)
    return prices, states, artifacts


def entropy_from_counts(counts: np.ndarray) -> float:
    probs = counts / counts.sum()
    return float(-(probs * np.log2(probs)).sum())


def summarize_protocol(asset: str, protocol: str, prices: np.ndarray, states: pd.DataFrame) -> dict:
    counts = states["pattern"].value_counts().sort_index()
    pattern_length_pairs = states[["pattern", "length"]].drop_duplicates()
    exact_block_replay = np.nan
    if protocol == "fixed_blocks_argmax":
        seqs = []
        for _, group in states.groupby("block"):
            seqs.append(group[["pattern", "length"]].to_numpy(dtype=int))
        exact_block_replay = bool(all(np.array_equal(seqs[0], seq) for seq in seqs[1:]))

    return {
        "asset": asset,
        "protocol": protocol,
        "segments": int(len(states)),
        "unique_patterns": int(len(counts)),
        "unique_pattern_length_pairs": int(len(pattern_length_pairs)),
        "dominant_pattern_share": float(counts.max() / counts.sum()),
        "pattern_entropy_bits": entropy_from_counts(counts.to_numpy()),
        "price_first": float(prices[0]),
        "price_last": float(prices[-1]),
        "price_std": float(np.std(prices)),
        "exact_block_replay": exact_block_replay,
        "pattern_counts": json.dumps({int(k): int(v) for k, v in counts.items()}),
    }

[cell 3 completed in 0.03s]


## Run the Old/New Rollout Comparison

This is intentionally short: five 252-day blocks per asset. It is enough to expose replay/collapse while keeping the notebook cheap to rerun.

In [3]:
ASSETS = ["goog", "sp500"]
PROTOCOLS = ["fixed_blocks_argmax", "continuous_argmax", "continuous_sampled", "continuous_empirical"]

all_summaries = []
all_states = []
all_prices = []
artifact_rows = []

for asset in ASSETS:
    for protocol in PROTOCOLS:
        print(f"running {asset} / {protocol} ...", flush=True)
        prices, states, artifacts = run_protocol(asset, protocol, seed=42)
        summary = summarize_protocol(asset, protocol, prices, states)
        all_summaries.append(summary)

        states_out = states.copy()
        states_out["protocol"] = protocol
        all_states.append(states_out)

        all_prices.append(
            pd.DataFrame({"asset": asset, "protocol": protocol, "t": np.arange(len(prices)), "price": prices})
        )
        artifact_rows.append(
            {
                "asset": asset,
                "n_segments": artifacts["n_segments"],
                "n_train_segments": artifacts["n_train_segments"],
                "n_test_segments": artifacts["n_test_segments"],
            }
        )

summary_df = pd.DataFrame(all_summaries)
states_df = pd.concat(all_states, ignore_index=True)
prices_df = pd.concat(all_prices, ignore_index=True)
artifact_df = pd.DataFrame(artifact_rows).drop_duplicates().reset_index(drop=True)

summary_df.to_csv(OUT / "rollout_state_summary.csv", index=False)
states_df.to_csv(OUT / "rollout_states_long.csv", index=False)
prices_df.to_csv(OUT / "rollout_prices_long.csv", index=False)
artifact_df.to_csv(OUT / "asset_artifact_summary.csv", index=False)

pd.set_option("display.max_colwidth", 120)
print("\nAsset artifacts:")
print(artifact_df.to_string(index=False))
print("\nRollout summary:")
print(summary_df.drop(columns=["pattern_counts"]).to_string(index=False))

running goog / fixed_blocks_argmax ...
running goog / continuous_argmax ...
running goog / continuous_sampled ...
running goog / continuous_empirical ...
running sp500 / fixed_blocks_argmax ...
running sp500 / continuous_argmax ...
running sp500 / continuous_sampled ...
running sp500 / continuous_empirical ...

Asset artifacts:
asset  n_segments  n_train_segments  n_test_segments
 goog         258               206               52
sp500         703               562              141

Rollout summary:
asset             protocol  segments  unique_patterns  unique_pattern_length_pairs  dominant_pattern_share  pattern_entropy_bits  price_first  price_last  price_std exact_block_replay
 goog  fixed_blocks_argmax       125                4                            4                0.480000              1.400924    39.306999   43.342999   0.723304               True
 goog    continuous_argmax       126                4                            4                0.492063              1.117

[cell 5 completed in 17.48s]


## Prediction Check

Expected if the collapse is mainly caused by deterministic/reset PEM rollout:

1. `fixed_blocks_argmax` has exact block replay.
2. `continuous_argmax` uses far fewer motifs than the sampled/empirical debug variants.
3. Sampling the PEM distribution increases pattern entropy.

In [4]:
checks = []
for asset in ASSETS:
    rows = summary_df.set_index(["asset", "protocol"])
    fixed = rows.loc[(asset, "fixed_blocks_argmax")]
    argmax = rows.loc[(asset, "continuous_argmax")]
    sampled = rows.loc[(asset, "continuous_sampled")]
    empirical = rows.loc[(asset, "continuous_empirical")]

    checks.append(
        {
            "asset": asset,
            "fixed_blocks_replay_exactly": bool(fixed["exact_block_replay"]),
            "sampled_unique_patterns_minus_argmax": int(sampled["unique_patterns"] - argmax["unique_patterns"]),
            "sampled_entropy_minus_argmax": float(sampled["pattern_entropy_bits"] - argmax["pattern_entropy_bits"]),
            "empirical_unique_patterns_minus_argmax": int(empirical["unique_patterns"] - argmax["unique_patterns"]),
        }
    )

checks_df = pd.DataFrame(checks)
checks_df.to_csv(OUT / "prediction_checks.csv", index=False)
print(checks_df.to_string(index=False))

assert checks_df["fixed_blocks_replay_exactly"].all()
assert (checks_df["sampled_unique_patterns_minus_argmax"] > 0).all()
assert (checks_df["sampled_entropy_minus_argmax"] > 0).all()
assert (checks_df["empirical_unique_patterns_minus_argmax"] > 0).all()

diagnosis = {
    "verdict": "prediction_confirmed",
    "interpretation": (
        "The author-style fixed-block protocol replays identical pattern-length paths, and deterministic "
        "argmax rollouts collapse to a small motif subset. Sampling PEM pattern/length distributions, or "
        "using an empirical Markov transition baseline, restores broad motif diversity."
    ),
    "outputs": {
        "summary": "rollout_state_summary.csv",
        "states": "rollout_states_long.csv",
        "prices": "rollout_prices_long.csv",
        "checks": "prediction_checks.csv",
    },
}
(OUT / "diagnosis.json").write_text(json.dumps(diagnosis, indent=2))
print(json.dumps(diagnosis, indent=2))

asset  fixed_blocks_replay_exactly  sampled_unique_patterns_minus_argmax  sampled_entropy_minus_argmax  empirical_unique_patterns_minus_argmax
 goog                         True                                     7                      2.201928                                       7
sp500                         True                                    11                      3.046331                                      12
{
  "verdict": "prediction_confirmed",
  "interpretation": "The author-style fixed-block protocol replays identical pattern-length paths, and deterministic argmax rollouts collapse to a small motif subset. Sampling PEM pattern/length distributions, or using an empirical Markov transition baseline, restores broad motif diversity.",
  "outputs": {
    "summary": "rollout_state_summary.csv",
    "states": "rollout_states_long.csv",
    "prices": "rollout_prices_long.csv",
    "checks": "prediction_checks.csv"
  }
}


[cell 7 completed in 0.00s]


## Visual Diagnostics

The state plots make the failure mode visible: fixed blocks restart and replay; argmax chains are narrow; sampled/empirical chains explore the motif vocabulary.

In [5]:
fig, axes = plt.subplots(len(ASSETS), 1, figsize=(11, 5.4), sharex=False)
if len(ASSETS) == 1:
    axes = [axes]
for ax, asset in zip(axes, ASSETS):
    for protocol in PROTOCOLS:
        sub = states_df[(states_df["asset"] == asset) & (states_df["protocol"] == protocol)].reset_index(drop=True)
        y_offset = PROTOCOLS.index(protocol) * (ASSET_CONFIG[asset]["k"] + 1)
        ax.plot(sub.index, sub["pattern"] + y_offset, marker=".", linewidth=1, label=protocol)
    ax.set_title(f"{ASSET_CONFIG[asset]['label']}: generated pattern sequence by protocol")
    ax.set_ylabel("pattern + protocol offset")
    ax.grid(alpha=0.25)
axes[-1].set_xlabel("generated segment index")
axes[0].legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout()
state_png = OUT / "pattern_sequence_by_protocol.png"
state_pdf = OUT / "pattern_sequence_by_protocol.pdf"
fig.savefig(state_png, dpi=220)
fig.savefig(state_pdf)
plt.close(fig)

fig, axes = plt.subplots(len(ASSETS), 1, figsize=(11, 5.4), sharex=True)
if len(ASSETS) == 1:
    axes = [axes]
for ax, asset in zip(axes, ASSETS):
    for protocol in PROTOCOLS:
        sub = prices_df[(prices_df["asset"] == asset) & (prices_df["protocol"] == protocol)]
        ax.plot(sub["t"], sub["price"], linewidth=1.3, label=protocol)
    ax.set_title(f"{ASSET_CONFIG[asset]['label']}: synthetic price path by protocol")
    ax.set_ylabel("price")
    ax.grid(alpha=0.25)
axes[-1].set_xlabel("generated day")
axes[0].legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout()
price_png = OUT / "price_path_by_protocol.png"
price_pdf = OUT / "price_path_by_protocol.pdf"
fig.savefig(price_png, dpi=220)
fig.savefig(price_pdf)
plt.close(fig)

print(f"saved {state_png.relative_to(ROOT)}")
print(f"saved {price_png.relative_to(ROOT)}")

saved reports/generated_outputs/12_rollout_debug_notebook/pattern_sequence_by_protocol.png
saved reports/generated_outputs/12_rollout_debug_notebook/price_path_by_protocol.png


[cell 9 completed in 0.41s]


## Notebook Conclusion

This does not prove the PGM is perfect. It does show that the most obvious generation collapse is already present in the state scheduler: deterministic `argmax` evolution plus fixed-state yearly restarts can replay or collapse the motif sequence before the decoder has much chance to help. A downstream debugging notebook should therefore inspect SISC artifacts, PEM transition probabilities, and PGM segments separately before interpreting TATR/TMTR curves.